In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from natsort import natsorted
import numpy as np
import re
import sys

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(rf'../../../../tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

tidy3dAPI = os.environ["API_TIDY3D_KEY"]




In [2]:
dir = rf"./data/diffraction_monitor_data"
os.makedirs(dir, exist_ok=True)

In [28]:
try:
    data_path = f"{dir}/20260708_average_diffraction_n_3.4_ff_0.237_ffh_0.185_schulz.h5"
    data_old = AM.read_hdf5_as_dict(data_path)
    print(data_old.keys())
except FileNotFoundError:
    print("File not found.")
    data_old = {}
except Exception as e:
    print(f"An error occurred: {e}")
    data_old = {}

folder_path = rf"../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction"

Error reading HDF5 file: [Errno 2] Unable to synchronously open file (unable to open file: name = './data/diffraction_monitor_data/20260708_average_diffraction_n_3.4_ff_0.237_ffh_0.185_schulz.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)
dict_keys([])


In [4]:
reference_object = AM.loadFromFile(key = tidy3dAPI, file_path=os.path.join(folder_path, "reference.txt"),get_ref=False)

amps_ref = reference_object.sim_data["diffraction"].amps
Pinc = np.abs(amps_ref.sel(orders_x=0, orders_y=0, polarization="p"))**2

reference_exit = reference_object.sim_data["flux1"].flux


Configured successfully.


14:44:33 W. Europe Daylight Time Billed flex credit cost: 0.109.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [5]:
n_values = list(data_old['n_values'] )if 'n_values' in data_old.keys() else []
ff_values =list( data_old['ff']) if 'ff' in data_old.keys() else []
size_values = list(data_old['sizes'])if 'sizes' in data_old.keys() else []
z_values = list(data_old['z_values']) if 'z_values' in data_old.keys() else []
values = data_old['transmission_data'] if 'transmission_data' in data_old.keys() else {}
reference_entry=None
# Loop through all files in the folder
for dirpath, dirnames, filenames in os.walk(folder_path):
      try:
        z_value = float(re.search(r'z_([+-]?\d+(?:\.\d+)?)', dirpath).group(1))
      except AttributeError:
        print(f"Could not extract z_value from directory: {dirpath}")
        continue
      
      z_values.append(z_value)
      for filename in filenames:
        try:
            n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff = float(re.search(r'ffr_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            size = float(re.search(r'size_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            sample = float(re.search(r'sample_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff_values.append(ff)
            n_values.append(n_value)
            size_values.append(size)
            try:
                test_val = values[n_value][ff][z_value][size][sample]
                print(f"Data for n={n_value}, ff={ff}, z={z_value}, size={size}, sample={sample} already exists. Skipping file: {filename}")
            except KeyError:
                #Retrieve simulation data 
                if os.path.isfile(os.path.join(dirpath, filename)):
                  file=os.path.join(dirpath, filename)
                  structure_1 = AM.loadFromFile(key = tidy3dAPI, file_path=file,get_ref=False)
                  sim_data_i = structure_1.sim_data
                  transmission_entry = sim_data_i['flux2'].flux
                  transmission_exit = sim_data_i['flux1'].flux
                 
                  if str(n_value) not in values.keys():
                    values[str(n_value)] = {}
                  if str(ff) not in values[str(n_value)].keys():
                      values[str(n_value)][str(ff)] = {}
                  if str(z_value) not in values[str(n_value)][str(ff)].keys():
                        values[str(n_value)][str(ff)][str(z_value)] = {}
                  if str(size) not in values[str(n_value)][str(ff)][str(z_value)].keys():
                        values[str(n_value)][str(ff)][str(z_value)][str(size)] = {}
                  if str(sample) not in values[str(n_value)][str(ff)][str(z_value)][str(size)].keys():
                        values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)] = {}

                  #p = co and s = cross
                  amps = structure_1.sim_data["diffraction"].amps
                  # T_co = abs(amps.sel(polarization="p"))**2/Pinc
                  # T_cross = abs(amps.sel(polarization="s"))**2/Pinc
                  # T_co_total = T_co.sum(dim=("orders_x", "orders_y"))
                  # T_cross_total = T_cross.sum(dim=("orders_x", "orders_y"))
                  # T_ballistic = np.abs(amps.sel(orders_x=0, orders_y=0, polarization="p")/amps_ref.sel(orders_x=0, orders_y=0, polarization="p"))**2
                  lambdas = td.C_0 / structure_1.sim_data["diffraction"].f
                  T_total = transmission_exit/reference_exit

                  values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)]["amps"] = amps
                  values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)]["T_total"] = T_total

        except Exception as e:
            print("Error:", e)
            continue
       






Could not extract z_value from directory: ../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction
Could not extract z_value from directory: ../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction\n_3.40
Configured successfully.


14:44:39 W. Europe Daylight Time Billed flex credit cost: 3.735.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

Configured successfully.


14:44:45 W. Europe Daylight Time Billed flex credit cost: 3.735.

...

14:56:30 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

Configured successfully.


14:56:35 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [34]:
values_processed = {}
for n in values.keys():
    print(f"Processing n={n}")
    for ff in values[n].keys():
        for z_value in values[n][ff].keys():
            for size in values[n][ff][z_value].keys():
                samples   = list(values[n][ff][z_value][size].keys())
                amps_list = [values[n][ff][z_value][size][s]["amps"] for s in samples]
                T_total_average = np.mean([values[n][ff][z_value][size][s]["T_total"] for s in samples], axis=0)
                N         = len(amps_list)

                # coherent amplitude average (for the ballistic term) -> DataArray
                amps_average = sum(amps_list) / N

                # incoherent intensity averages ⟨|a|²⟩ -> DataArray (.sel/.sum still work)
                T_co    = sum(abs(a.sel(polarization="p"))**2 for a in amps_list) / N / Pinc
                T_cross = sum(abs(a.sel(polarization="s"))**2 for a in amps_list) / N / Pinc

                T_co_total    = T_co.sum(dim=("orders_x", "orders_y"))
                T_cross_total = T_cross.sum(dim=("orders_x", "orders_y"))

                T_ballistic = np.abs(
                    amps_average.sel(orders_x=0, orders_y=0, polarization="p"))**2/ Pinc
                
                if str(n_value) not in values_processed.keys():
                    values_processed[str(n_value)] = {}
                if str(ff) not in values_processed[str(n_value)].keys():
                    values_processed[str(n_value)][str(ff)] = {}
                if str(z_value) not in values_processed[str(n_value)][str(ff)].keys():
                      values_processed[str(n_value)][str(ff)][str(z_value)] = {}
                if str(size) not in values_processed[str(n_value)][str(ff)][str(z_value)].keys():
                      values_processed[str(n_value)][str(ff)][str(z_value)][str(size)] = {}
                    
                values_processed[str(n_value)][str(ff)][str(z_value)][str(size)]["T_ballistic"] = T_ballistic
                values_processed[str(n_value)][str(ff)][str(z_value)][str(size)]["T_co"] = T_co_total
                values_processed[str(n_value)][str(ff)][str(z_value)][str(size)]["T_cross"] = T_cross_total
                values_processed[str(n_value)][str(ff)][str(z_value)][str(size)]["T_total"] = T_total_average




Processing n=3.4


In [35]:
# After the loop, get unique values as arrays
n_unique = np.unique(n_values)
ff_unique = np.unique(ff_values)
size_unique = np.unique(size_values)
z_unique = np.unique(z_values)

In [36]:
values_processed['3.4']['0.2369'].keys()

dict_keys(['100.0', '5.0'])

In [37]:

data = {
    "transmission_data": values_processed,
    "n_values": n_unique,
    "ff_values": ff_unique,
    "size_values": size_unique,
    "z_values": z_unique,
    "lambdas": lambdas
}

In [38]:
create_hdf5_from_dict(data,data_path)